In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
veg_price = pd.read_csv('data/fruitvegprices-2017_2022.csv')
veg_price

In [ ]:
uk_weather = pd.read_csv('data/combined_united_kingdom_2017_to_2022.csv')
uk_weather

In [ ]:
sns.set(rc={"figure.figsize":(20,12)})
sns.lineplot(data=veg_price, x='date', y='price', hue='variety')

In [ ]:
veg_price_apple = veg_price[veg_price['item'] == 'apples']
veg_price_apple

In [ ]:
veg_price_pear = veg_price[veg_price['item'] == 'pears']
veg_price_pear

In [ ]:
sns.set(rc={"figure.figsize":(20,12)})
sns.lineplot(data=veg_price_pear, x='date', y='price', hue='variety')

In [ ]:
uk_weather['datetime'] = pd.to_datetime(uk_weather['datetime'])

start_date = '2017-11-03'
end_date = '2022-03-11'

filtered_data = uk_weather[(uk_weather['datetime'] >= start_date) & (uk_weather['datetime'] <= end_date)]

uk_weatherweakly = filtered_data[filtered_data['datetime'].isin(pd.date_range(start=start_date, end=filtered_data['datetime'].max(), freq='7D'))]

uk_weatherweakly

In [ ]:
sns.set(rc={"figure.figsize":(20,12)})
sns.lineplot(data=uk_weatherweakly, x='datetime', y='temp')

In [ ]:
uk_weatherweakly.isna().sum()


In [ ]:
veg_price_pear.isna().sum()

In [ ]:
veg_price_pear ['date'] = pd.to_datetime(veg_price_pear['date'])
veg_price_pear

In [ ]:
veg_price_pear['date'] = pd.to_datetime(veg_price_pear['date'])
merged_data = pd.merge(veg_price_pear, uk_weatherweakly, left_on='date', right_on='datetime')
X = merged_data[['temp']]
y = merged_data['price']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)

In [ ]:
y_pred = knn.predict(X_test)

In [ ]:
mse = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error: {mse}")

In [ ]:
test_dates = merged_data.loc[X_test.index, 'date']

# Create a DataFrame to hold the true and predicted prices with dates
results_df = pd.DataFrame({
    'Date': test_dates,
    'Actual Price': y_test,
    'Predicted Price': y_pred
})

In [ ]:
sns.set(rc={"figure.figsize":(10,6)})
sns.lineplot(data=results_df, x='Date', y='Actual Price', label='True Prices')
sns.lineplot(data=results_df, x='Date', y='Predicted Price', label='Predicted Prices', linestyle='--')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load and merge datasets
prices_data = pd.read_csv('data/fruitvegprices-2017_2022.csv')
weather_data = pd.read_csv('data/combined_united_kingdom_2017_to_2022.csv')

# Standardize column names and merge on 'datetime'
prices_data.rename(columns={'date': 'datetime'}, inplace=True)
prices_data['datetime'] = pd.to_datetime(prices_data['datetime'])
weather_data['datetime'] = pd.to_datetime(weather_data['datetime'])

# Check unique products in prices_data
print("Unique products in prices data:", prices_data['item'].unique())

# Filter to include only certain products and merge datasets
products_to_analyze = ['apples', 'bananas', 'carrots']  # Make lowercase
prices_data['item'] = prices_data['item'].str.lower().str.strip()
filtered_prices = prices_data[prices_data['item'].isin(products_to_analyze)]

# Merge datasets
merged_data = pd.merge(filtered_prices, weather_data, on='datetime', how='inner')

# Confirm available data for each product
for product in products_to_analyze:
    count = merged_data[merged_data['item'] == product].shape[0]
    print(f"Data points available for {product.capitalize()}: {count}")

# Perform regression for each product and feature
weather_features = ['temp', 'humidity', 'windspeed']

for product in products_to_analyze:
    product_data = merged_data[merged_data['item'] == product]

    if len(product_data) < 10:
        print(f"Not enough data for {product.capitalize()}. Skipping.")
        continue

    for feature in weather_features:
        X = product_data[[feature]]
        y = product_data['price']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        model = LinearRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        print(f"{product.capitalize()} - {feature.capitalize()} - MSE: {mse:.2f}, R^2: {r2:.2f}")

        plt.figure(figsize=(8, 5))
        plt.scatter(X_test, y_test, color='blue', label='Actual')
        plt.plot(X_test, y_pred, color='red', label='Predicted')
        plt.xlabel(feature.capitalize())
        plt.ylabel('Price')
        plt.title(f'{product.capitalize()} Price vs {feature.capitalize()}')
        plt.legend()
        plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load and merge datasets
prices_data = pd.read_csv('data/fruitvegprices-2017_2022.csv')
weather_data = pd.read_csv('data/combined_united_kingdom_2017_to_2022.csv')

# Standardize column names and merge on 'datetime'
prices_data.rename(columns={'date': 'datetime'}, inplace=True)
prices_data['datetime'] = pd.to_datetime(prices_data['datetime'])
weather_data['datetime'] = pd.to_datetime(weather_data['datetime'])

# Check unique products in prices_data
print("Unique products in prices data:", prices_data['item'].unique())

# Filter to include only certain products and merge datasets
products_to_analyze = ['apples', 'bananas', 'carrots']  # Make lowercase
prices_data['item'] = prices_data['item'].str.lower().str.strip()
filtered_prices = prices_data[prices_data['item'].isin(products_to_analyze)]

# Merge datasets
merged_data = pd.merge(filtered_prices, weather_data, on='datetime', how='inner')

# Confirm available data for each product
for product in products_to_analyze:
    count = merged_data[merged_data['item'] == product].shape[0]
    print(f"Data points available for {product.capitalize()}: {count}")

# Original weather features list
weather_features = ['temp', 'humidity', 'windspeed']

# Exclude 'humidity' and define weather_features_tw
weather_features_tw = [feature for feature in weather_features if feature != 'humidity']

# Perform regression for each product and feature in weather_features_tw
for product in products_to_analyze:
    product_data = merged_data[merged_data['item'] == product]

    if len(product_data) < 10:
        print(f"Not enough data for {product.capitalize()}. Skipping.")
        continue

    for feature in weather_features_tw:
        X = product_data[[feature]]
        y = product_data['price']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        model = LinearRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        print(f"{product.capitalize()} - {feature.capitalize()} - MSE: {mse:.2f}, R^2: {r2:.2f}")

        plt.figure(figsize=(8, 5))
        plt.scatter(X_test, y_test, color='blue', label='Actual')
        plt.plot(X_test, y_pred, color='red', label='Predicted')
        plt.xlabel(feature.capitalize())
        plt.ylabel('Price')
        plt.title(f'{product.capitalize()} Price vs {feature.capitalize()}')
        plt.legend()
        plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load and merge datasets
prices_data = pd.read_csv('data/fruitvegprices-2017_2022.csv')
weather_data = pd.read_csv('data/combined_united_kingdom_2017_to_2022.csv')

# Standardize column names and merge on 'datetime'
prices_data.rename(columns={'date': 'datetime'}, inplace=True)
prices_data['datetime'] = pd.to_datetime(prices_data['datetime'])
weather_data['datetime'] = pd.to_datetime(weather_data['datetime'])

# Check unique products in prices_data
print("Unique products in prices data:", prices_data['item'].unique())

# Filter to include only certain products and merge datasets
products_to_analyze = ['apples', 'bananas', 'carrots']  # Make lowercase
prices_data['item'] = prices_data['item'].str.lower().str.strip()
filtered_prices = prices_data[prices_data['item'].isin(products_to_analyze)]

# Merge datasets
merged_data = pd.merge(filtered_prices, weather_data, on='datetime', how='inner')

# Confirm available data for each product
for product in products_to_analyze:
    count = merged_data[merged_data['item'] == product].shape[0]
    print(f"Data points available for {product.capitalize()}: {count}")

# Original weather features list
weather_features = ['temp', 'humidity', 'windspeed']

# Define a new list with only 'humidity' and 'windspeed'
weather_features_hw = ['humidity', 'windspeed']

# Perform regression for each product and feature in weather_features_hw
for product in products_to_analyze:
    product_data = merged_data[merged_data['item'] == product]

    if len(product_data) < 10:
        print(f"Not enough data for {product.capitalize()}. Skipping.")
        continue

    for feature in weather_features_hw:
        X = product_data[[feature]]
        y = product_data['price']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        model = LinearRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        mse = mean_squared_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)

        print(f"{product.capitalize()} - {feature.capitalize()} - MSE: {mse:.2f}, R^2: {r2:.2f}")

        plt.figure(figsize=(8, 5))
        plt.scatter(X_test, y_test, color='blue', label='Actual')
        plt.plot(X_test, y_pred, color='red', label='Predicted')
        plt.xlabel(feature.capitalize())
        plt.ylabel('Price')
        plt.title(f'{product.capitalize()} Price vs {feature.capitalize()}')
        plt.legend()
        plt.show()


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Perform regression using both 'temp' and 'humidity' as predictors
for product in products_to_analyze:
    product_data = merged_data[merged_data['item'] == product]

    if len(product_data) < 10:
        print(f"Not enough data for {product.capitalize()}. Skipping.")
        continue

    # Define predictors and target
    X = product_data[['temp', 'humidity']]
    y = product_data['price']

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train the regression model
    model = LinearRegression()
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"{product.capitalize()} - Temp & Humidity - MSE: {mse:.2f}, R^2: {r2:.2f}")

    # 3D Scatter Plot: Actual vs. Predicted
    fig = plt.figure(figsize=(10, 7))
    ax = fig.add_subplot(111, projection='3d')

    # Scatter points: Actual data
    ax.scatter(X_test['temp'], X_test['humidity'], y_test, color='blue', label='Actual Prices')

    # Line: Predicted data
    ax.scatter(X_test['temp'], X_test['humidity'], y_pred, color='red', label='Predicted Prices')

    ax.set_xlabel('Temperature')
    ax.set_ylabel('Humidity')
    ax.set_zlabel('Price')
    ax.set_title(f'{product.capitalize()} Price Prediction (Temp & Humidity)')
    ax.legend()
    plt.show()
